# 02B Homework 04 ETM — Graphs (Old Layout)

Same graph-building pipeline as `02_Homework04_Graphs.ipynb`, run on the **Old Layout**
geometry in `Homework04/Objects/Old Layout/`. The old layout additionally contains a
**Dining** room (absent from the current layout).

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [ ]:
renderer = "vscode"

# Shared graph styling (kept consistent across notebook graphs)
GRAPH_NODE_SIZE_KEY = "size"
GRAPH_NODE_COLOR_KEY = "color"
GRAPH_NODE_LABEL_KEY = "label"
GRAPH_EDGE_COLOR = "#6F6F6F"
GRAPH_EDGE_WIDTH = 3
GRAPH_BG_DARK = "black"
GRAPH_BG_LIGHT = "white"

In [ ]:
import os
os.makedirs("Images", exist_ok=True)

## 4. Load Room Geometry

Each room type (Bedroom, Bathroom, Corridor, Kitchen, Living room, Stair, Balcony) is loaded from its OBJ file and built into a closed volumetric `Cell`. A selector vertex at the cell interior stores the room's type, colour, and label. Rooms that cannot be resolved into a closed cell fall back to a face-centroid selector.

In [ ]:
BASE = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout"

ROOM_TYPES = {
    "Bedroom":     {"path": BASE + r"\Bedroom.obj",     "color": "#E63946"},
    "Bathroom":    {"path": BASE + r"\Bathroom.obj",    "color": "#4CC9F0"},
    "Corridor":    {"path": BASE + r"\Corridor.obj",    "color": "#457B9D"},
    "Kitchen":     {"path": BASE + r"\Kitchen.obj",     "color": "#F4A261"},
    "Living room": {"path": BASE + r"\Living room.obj", "color": "#FFBE0B"},
    "Stair":       {"path": BASE + r"\Stair.obj",       "color": "#8338EC"},
    "Balcony":     {"path": BASE + r"\balcony.obj",     "color": "#2CA02C"},
    "Storeroom":   {"path": BASE + r"\Storeroom.obj",   "color": "#8D6E63"},
    "Dining":      {"path": BASE + r"\Dining.obj",      "color": "#76B7B2"},
}

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)**0.5

def get_val(topology, key):
    d = Topology.Dictionary(topology)
    if d is None:
        return None
    v = Dictionary.ValueAtKey(d, key)
    if isinstance(v, list):
        return v[0] if v else None
    return v

def try_build_cell(faces):
    """Try Cell.ByFaces at progressively looser tolerances; return first success."""
    for face_set in [faces, faces]:  # try same set; different tols
        for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
            c = Cell.ByFaces(face_set, tolerance=tol)
            if c is not None:
                return c
    return None

all_faces_raw   = []
all_faces_clean = []
selector        = []
cells           = []

for type_name, info in ROOM_TYPES.items():
    objs = Topology.ByOBJPath(info["path"], selfMerge=False)
    if not isinstance(objs, list):
        objs = [objs]

    count = 0
    for i, obj in enumerate(objs):
        raw_faces = Topology.Faces(obj) or []
        if not raw_faces:
            continue
        all_faces_raw.extend(raw_faces)

        cleaned_obj = Topology.RemoveCoplanarFaces(obj, epsilon=0.1, tolerance=0.001, silent=True)
        clean_obj   = cleaned_obj if cleaned_obj else obj
        clean_faces = Topology.Faces(clean_obj) or []
        all_faces_clean.extend(clean_faces)

        d = Dictionary.ByKeysValues(
            ["color", "type", "label", "vertex_size"],
            [info["color"], type_name, type_name, 20]
        )

        # Try raw faces first, then cleaned, at multiple tolerances
        c = try_build_cell(raw_faces) or try_build_cell(clean_faces)

        if c is not None:
            c2 = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
            c  = c2 if c2 else c
            c  = Topology.RemoveCollinearEdges(c) or c
            s  = Topology.InternalVertex(c)
            c  = Topology.SetDictionary(c, d)
            cells.append(c)
        else:
            # Centroid fallback so the selector always exists
            cen = Topology.Centroid(Cluster.ByTopologies(raw_faces))
            s   = Vertex.ByCoordinates(cen.X(), cen.Y(), cen.Z())
            print(f"  fallback selector: {type_name} #{i+1}")

        selector.append(Topology.SetDictionary(s, d))
        count += 1

    print(f"{type_name}: {count} object(s)  |  cells so far: {len(cells)}")

print(f"\nTotal selectors : {len(selector)}")
print(f"Total cells     : {len(cells)}")
print(f"Raw faces       : {len(all_faces_raw)}")

## 5. Visualize Room Volumes

In [ ]:
Topology.Show(
    cells,
    selector,
    faceColorKey="color",
    faceOpacity=0.4,
    showEdges=True,
    edgeWidth=3,
    showVertices=True,
    vertexSize=10,
    vertexLabelKey="label",
    showVertexLabel=True,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 6. Compute Face Adjacency

Two rooms are treated as adjacent when they share either a vertical wall face or a horizontal floor/ceiling face, based on face normals and overlapping face bounding boxes.

An open-space pair is recorded only for non-adjacent rooms with opposing vertical wall faces within 10 cm, indicating a visually open connection with no separating wall.

In [ ]:
from collections import Counter
import math

# -- Per-cell face data ---------------------------------------------------------
cell_face_cents  = []   # raw (x,y,z) centroids
cell_face_planes = []   # (centroid_xyz, unit_normal)
cell_face_data   = []   # (centroid_xyz, unit_normal, bbox_xyz)
for c in cells:
    fcts = []
    fpls = []
    fdat = []
    for f in (Topology.Faces(c) or []):
        fc = Topology.Centroid(f)
        fp = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn = Face.Normal(f)
        fvs = Topology.Vertices(f) or []
        if fvs:
            xs = [Vertex.X(v) for v in fvs]
            ys = [Vertex.Y(v) for v in fvs]
            zs = [Vertex.Z(v) for v in fvs]
            bbox = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bbox = None
        fcts.append(fp)
        fpls.append((fp, fn))
        fdat.append((fp, fn, bbox))
    cell_face_cents.append(fcts)
    cell_face_planes.append(fpls)
    cell_face_data.append(fdat)

# Vertical axis in this model is Z (index 2).
UP_AXIS = 2

# -- Face contact helpers -------------------------------------------------------
def axis_ranges(bbox):
    return ((bbox[0], bbox[1]), (bbox[2], bbox[3]), (bbox[4], bbox[5]))

def interval_overlap(a, b):
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))

def interval_gap(a, b):
    return max(0.0, max(a[0], b[0]) - min(a[1], b[1]))

def dominant_axis(fn):
    return max(range(3), key=lambda k: abs(fn[k]))

def dot3(a, b):
    return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]

def perp_dist(pt, fp, fn):
    dx, dy, dz = pt[0]-fp[0], pt[1]-fp[1], pt[2]-fp[2]
    return abs(fn[0]*dx + fn[1]*dy + fn[2]*dz)

def is_floor_ceil(fn):
    return abs(fn[UP_AXIS]) > 0.7

def faces_contact(fa, fb, mode):
    fp_i, fn_i, bb_i = fa
    fp_j, fn_j, bb_j = fb
    if bb_i is None or bb_j is None:
        return False

    # Opposite-facing surfaces are expected for a shared boundary.
    if dot3(fn_i, fn_j) > -0.65:
        return False

    ai = dominant_axis(fn_i)
    aj = dominant_axis(fn_j)

    if mode == "wall":
        if abs(fn_i[UP_AXIS]) >= 0.7 or abs(fn_j[UP_AXIS]) >= 0.7:
            return False
        if ai == UP_AXIS or aj == UP_AXIS:
            return False
        if ai != aj:
            return False
        n_axis = ai
        tangential_axes = [ax for ax in range(3) if ax != n_axis]
        gap_tol = 0.20
        ov_tol_0 = 0.20
        ov_tol_1 = 0.20
    else:
        if abs(fn_i[UP_AXIS]) < 0.7 or abs(fn_j[UP_AXIS]) < 0.7:
            return False
        if ai != UP_AXIS or aj != UP_AXIS:
            return False
        n_axis = UP_AXIS
        tangential_axes = [0, 1]
        gap_tol = 0.08
        ov_tol_0 = 0.80
        ov_tol_1 = 0.80

    ri = axis_ranges(bb_i)
    rj = axis_ranges(bb_j)

    gap_n = interval_gap(ri[n_axis], rj[n_axis])
    ov_0 = interval_overlap(ri[tangential_axes[0]], rj[tangential_axes[0]])
    ov_1 = interval_overlap(ri[tangential_axes[1]], rj[tangential_axes[1]])

    if gap_n > gap_tol:
        return False
    if ov_0 < ov_tol_0 or ov_1 < ov_tol_1:
        return False
    if mode != "wall" and (ov_0 * ov_1) < 1.0:
        return False

    return True

def rooms_share_wall(i, j):
    for fa in cell_face_data[i]:
        for fb in cell_face_data[j]:
            if faces_contact(fa, fb, mode="wall"):
                return True
    return False

def rooms_share_floor_ceil(i, j):
    for fa in cell_face_data[i]:
        for fb in cell_face_data[j]:
            if faces_contact(fa, fb, mode="horiz"):
                return True
    return False

# -- Adjacency list: direct shared boundary contact -----------------------------
adj_pairs = []
for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        if rooms_share_wall(i, j) or rooms_share_floor_ceil(i, j):
            adj_pairs.append((i, j))
adj_set = set(adj_pairs)

# -- Open-space adjacency: opposing vertical wall faces within 10 cm ------------
open_pairs = []
for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        if (i, j) in adj_set:
            continue
        found = False
        for fp_i, fn_i in cell_face_planes[i]:
            if is_floor_ceil(fn_i):
                continue
            for fp_j, fn_j in cell_face_planes[j]:
                if is_floor_ceil(fn_j):
                    continue
                if dot3(fn_i, fn_j) > -0.85:
                    continue
                if perp_dist(fp_i, fp_j, fn_j) >= 0.1:
                    continue
                if dist3(fp_i, fp_j) < 1.0:
                    open_pairs.append((i, j))
                    found = True
                    break
            if found:
                break

# -- Area-scaled vertex sizes --------------------------------------------------
areas    = [Cell.SurfaceArea(c) for c in cells]
min_area = min(areas)
max_area = max(areas)

for i, (s, area) in enumerate(zip(selector, areas)):
    size  = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))
    d_old = Topology.Dictionary(s)
    keys  = list(Dictionary.Keys(d_old)) + ["area", "vertex_size"]
    vals  = list(Dictionary.Values(d_old)) + [area, size]
    Topology.SetDictionary(s, Dictionary.ByKeysValues(keys, vals))

cell_cluster = Cluster.ByTopologies(cells)

print(f"Individual cells  : {len(cells)}")
print(f"Adjacent pairs    : {len(adj_pairs)}")
print(f"Open-space pairs  : {len(open_pairs)}")

types_found = Counter(get_val(s, "type") or "unknown" for s in selector)
print("\nRoom type breakdown:")
for t, n in sorted(types_found.items()):
    print(f"  {t}: {n}")

## 7. Merge Stair Cells

The staircase is made of several cells across floors. They are grouped by **plan footprint** `(x, y)`: the cells that share one footprint form the main multi-floor shaft, and the remaining cell(s) at a different footprint form the smaller entrance stair. Each group is collapsed into one representative node placed at the average centroid of its members, with area-scaled size so the smaller group gets a visually smaller node. Grouping by footprint (rather than by object order) keeps each representative node sitting on its actual room volume even if the number or order of objects inside `Stair.obj` changes.

- `merged_nodes` — all non-stair room nodes + `stair_main_rep` + `stair_ext_rep`
- `to_merged(i)` — maps any original cell index to its merged node index
- `merged_size(mi)` — returns an area-scaled vertex size for merged node `mi`

In [ ]:
# ── Merge stair cells into two representative nodes ────────────────────────
# Old-layout stairs are not a single stacked shaft, so exact-footprint grouping fails.
# Instead cluster stair cells by plan (X, Y) proximity (single-linkage): a shaft that
# shifts in plan between floors still groups together, while a spatially separate entrance
# stair forms its own cluster. The cluster with the largest TOTAL AREA is the main/large
# stair; the remaining cluster(s) form the small entrance stair.
from collections import defaultdict

stair_idxs = [i for i, s in enumerate(selector) if get_val(s, "type") == "Stair"]

XY_CLUSTER_TOL = 5.0  # plan distance (m) below which two stair cells are the same stair
def _xy(i):
    return (Vertex.X(selector[i]), Vertex.Y(selector[i]))
def _xy_close(i, j):
    return (_xy(i)[0]-_xy(j)[0])**2 + (_xy(i)[1]-_xy(j)[1])**2 <= XY_CLUSTER_TOL**2

clusters = []
for i in stair_idxs:
    placed = False
    for cl in clusters:
        if any(_xy_close(i, j) for j in cl):
            cl.append(i); placed = True; break
    if not placed:
        clusters.append([i])
# Merge transitively-linked clusters.
_changed = True
while _changed:
    _changed = False
    for a in range(len(clusters)):
        for b in range(a + 1, len(clusters)):
            if any(_xy_close(i, j) for i in clusters[a] for j in clusters[b]):
                clusters[a] += clusters[b]; del clusters[b]; _changed = True; break
        if _changed:
            break

clusters.sort(key=lambda cl: sum(areas[i] for i in cl), reverse=True)
stair_main_set = set(clusters[0])                               # largest area = main/large stair
stair_ext_set  = set(i for cl in clusters[1:] for i in cl)      # the rest = entrance stair

# Guard: if every stair fell into one cluster, peel off its lowest cell as the entrance
# node so downstream code always has two distinct stair nodes.
if not stair_ext_set:
    lowest = min(stair_main_set, key=lambda i: Vertex.Z(selector[i]))
    stair_ext_set = {lowest}
    stair_main_set = (stair_main_set - {lowest}) or {lowest}

non_stair_idxs = sorted(i for i in range(len(cells)) if i not in set(stair_idxs))

def _stair_centroid(idx_set):
    n = len(idx_set)
    return (
        sum(Vertex.X(selector[i]) for i in idx_set) / n,
        sum(Vertex.Y(selector[i]) for i in idx_set) / n,
        sum(Vertex.Z(selector[i]) for i in idx_set) / n,
    )

sx_m, sy_m, sz_m = _stair_centroid(stair_main_set)
stair_main_rep = Vertex.ByCoordinates(sx_m, sy_m, sz_m)
stair_main_rep = Topology.SetDictionary(stair_main_rep, Dictionary.ByKeysValues(
    ["color", "type", "label", "area"],
    ["#8338EC", "Stair", "Stair", sum(areas[i] for i in stair_main_set)]
))

sx_e, sy_e, sz_e = _stair_centroid(stair_ext_set)
stair_ext_rep = Vertex.ByCoordinates(sx_e, sy_e, sz_e)
stair_ext_rep = Topology.SetDictionary(stair_ext_rep, Dictionary.ByKeysValues(
    ["color", "type", "label", "area"],
    ["#8338EC", "Stair", "Stair", sum(areas[i] for i in stair_ext_set)]
))

merged_nodes   = [selector[i] for i in non_stair_idxs] + [stair_main_rep, stair_ext_rep]
stair_main_idx = len(merged_nodes) - 2   # large / main stair
stair_ext_idx  = len(merged_nodes) - 1   # small entrance stair

_idx_map = {orig: new for new, orig in enumerate(non_stair_idxs)}
_idx_map.update({i: stair_main_idx for i in stair_main_set})
_idx_map.update({i: stair_ext_idx  for i in stair_ext_set})
def to_merged(cell_idx): return _idx_map[cell_idx]

merged_areas    = (
    [areas[i] for i in non_stair_idxs]
    + [sum(areas[i] for i in stair_main_set)]
    + [sum(areas[i] for i in stair_ext_set)]
)
merged_min_area = min(merged_areas)
merged_max_area = max(merged_areas)

def merged_size(mi):
    return 12 + int(48 * (merged_areas[mi] - merged_min_area) / (merged_max_area - merged_min_area + 1))

print(f"Main/large stair : {len(stair_main_set)} cell(s) {sorted(stair_main_set)} → 1 node  "
      f"(centroid x,y={round(sx_m,1),round(sy_m,1)}, area={sum(areas[i] for i in stair_main_set):.1f})")
print(f"Entrance stair   : {len(stair_ext_set)} cell(s) {sorted(stair_ext_set)} → 1 node  "
      f"(centroid x,y={round(sx_e,1),round(sy_e,1)}, area={sum(areas[i] for i in stair_ext_set):.1f})")
print(f"Graph nodes total : {len(merged_nodes)}  ({len(non_stair_idxs)} rooms + 2 stair nodes)")

## 8. Primal Graph

The primal graph represents the raw architectural topology: every geometric corner of the room volumes is a vertex, and every boundary edge (walls, floors, ceilings) shared between cells is an edge.

In [ ]:
primal_verts = Topology.Vertices(cell_cluster) or []
primal_edges = Topology.Edges(cell_cluster) or []
print(f"Primal - Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

Topology.Show(
    cell_cluster,
    faceColor=[210, 210, 250], faceOpacity=0.15,
    edgeColor=GRAPH_EDGE_COLOR, edgeWidth=GRAPH_EDGE_WIDTH,
    vertexColor=GRAPH_EDGE_COLOR, vertexSize=6,
    showVertices=True,
    backgroundColor=GRAPH_BG_DARK,
    width=800, height=600,
    renderer=renderer
)

## 9. Horizontal Adjacency - Wall Dual Graph

One node per room; stair cells are collapsed into two nodes (main 4-floor shaft + extra elements, see §7).

Two rooms are connected by an edge if they share a vertical wall face. Height is measured on the Z axis, so floor/ceiling faces are excluded from this graph.

In [ ]:
# -- Horizontal adjacency: rooms sharing a vertical wall face -------------------
dual_pair_set = set()
for i, j in adj_pairs:
    if not rooms_share_wall(i, j):
        continue
    gi, gj = to_merged(i), to_merged(j)
    if gi != gj:
        dual_pair_set.add((min(gi, gj), max(gi, gj)))

dual_edges = [Edge.ByVertices([merged_nodes[i], merged_nodes[j]]) for i, j in sorted(dual_pair_set)]
g_dual = Graph.ByVerticesEdges(merged_nodes, dual_edges)
print(f"Horizontal adjacency - Nodes: {len(Graph.Vertices(g_dual))}, Edges: {len(Graph.Edges(g_dual))}")

for v in Graph.Vertices(g_dual) or []:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    mi = min(range(len(merged_nodes)), key=lambda k: dist3(
        (vx, vy, vz),
        (Vertex.X(merged_nodes[k]), Vertex.Y(merged_nodes[k]), Vertex.Z(merged_nodes[k]))
    ))
    d_old = Topology.Dictionary(merged_nodes[mi])
    keys = list(Dictionary.Keys(d_old)) if d_old else []
    vals = list(Dictionary.Values(d_old)) if d_old else []
    if "size" in keys:
        vals[keys.index("size")] = merged_size(mi)
    else:
        keys.append("size")
        vals.append(merged_size(mi))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(keys, vals))
for e in Graph.Edges(g_dual) or []:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [GRAPH_EDGE_WIDTH, GRAPH_EDGE_COLOR]))

Topology.Show(
    g_dual, cell_cluster,
    faceColorKey="color", faceOpacity=0.05,
    edgeColor=GRAPH_EDGE_COLOR, edgeWidth=0.5,
    vertexSizeKey=GRAPH_NODE_SIZE_KEY, vertexColorKey=GRAPH_NODE_COLOR_KEY,
    vertexLabelKey=GRAPH_NODE_LABEL_KEY, showVertexLabel=True, showVertices=True,
    edgeWidthKey="width", edgeColorKey="color",
    backgroundColor=GRAPH_BG_DARK, width=800, height=600,
    renderer=renderer
)

## 10. Load Windows

In [ ]:
WIN_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout\window.obj"

win_objects = Topology.ByOBJPath(WIN_PATH, selfMerge=False)
if not isinstance(win_objects, list):
    win_objects = [win_objects]

windows = []
for obj in win_objects:
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is not None:
            windows.append(f)

for w in windows:
    Topology.SetDictionary(w, Dictionary.ByKeysValues(["color", "type"], ["#4CC9F0", "window"]))

print(f"Windows found: {len(windows)}")
win_cluster = Cluster.ByTopologies(windows)

Topology.Show(
    cell_cluster, win_cluster,
    faceColor="#4CC9F0", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 11. Load Doors

In [ ]:
DOOR_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout\door.obj"

door_objects = Topology.ByOBJPath(DOOR_PATH, selfMerge=False)
if not isinstance(door_objects, list):
    door_objects = [door_objects]

doors = []
for oi, obj in enumerate(door_objects):
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is None:
            # Collinear or near-degenerate edge in wire — clean and retry
            w2 = Topology.RemoveCollinearEdges(w)
            f  = Face.ByWire(w2) if w2 else None
        if f is not None:
            doors.append(f)
        else:
            print(f"  Warning: door object #{oi+1} wire could not be built as face")

for d in doors:
    Topology.SetDictionary(d, Dictionary.ByKeysValues(["color", "type"], ["#F4A261", "door"]))

print(f"Doors found: {len(doors)}")
door_cluster = Cluster.ByTopologies(doors)

Topology.Show(
    cell_cluster, door_cluster,
    faceColor="#F4A261", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 12. Load Entrance Doors

The building's exterior entrance doors are loaded separately from interior doors so they can be coloured and counted distinctly (red). Each wire is built into a face; degenerate wires are cleaned and retried.

In [ ]:
ENTRANCE_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Old Layout\Entrance door.obj"

entrance_objects = Topology.ByOBJPath(ENTRANCE_PATH, selfMerge=False)
if not isinstance(entrance_objects, list):
    entrance_objects = [entrance_objects]

entrance_doors = []
for oi, obj in enumerate(entrance_objects):
    if obj is None:
        continue
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is None:
            # Collinear or near-degenerate edge in wire — clean and retry
            w2 = Topology.RemoveCollinearEdges(w)
            f  = Face.ByWire(w2) if w2 else None
        if f is not None:
            entrance_doors.append(f)
        else:
            print(f"  Warning: entrance door object #{oi+1} wire could not be built as face")

for d in entrance_doors:
    Topology.SetDictionary(d, Dictionary.ByKeysValues(["color", "type"], ["#E63946", "entrance"]))

print(f"Entrance doors found: {len(entrance_doors)}")
entrance_cluster = Cluster.ByTopologies(entrance_doors) if entrance_doors else None

if entrance_cluster is not None:
    Topology.Show(
        cell_cluster, entrance_cluster,
        faceColor="#E63946", faceOpacity=0.25,
        edgeColor="white", edgeWidth=0.5,
        showVertices=False,
        backgroundColor="black",
        width=800, height=600,
        renderer=renderer
    )

## 13. Primal Graph with Apertures

The primal graph overlaid on the room volumes together with all aperture faces: windows (blue) and doors (orange).

In [ ]:
aperture_cluster = Cluster.ByTopologies(windows + doors + entrance_doors)
primal_verts   = Topology.Vertices(cell_cluster) or []
primal_edges   = Topology.Edges(cell_cluster)    or []
primal_cluster = Cluster.ByTopologies(primal_edges + primal_verts)

print(f"Primal - Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

Topology.Show(
    cell_cluster, aperture_cluster, primal_cluster,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor=GRAPH_EDGE_COLOR, edgeWidth=GRAPH_EDGE_WIDTH,
    vertexColor=GRAPH_EDGE_COLOR, vertexSize=6,
    showVertices=True,
    backgroundColor=GRAPH_BG_DARK,
    width=900, height=700,
    renderer=renderer
)

## 14. Match Apertures to Rooms

Each aperture face (window, door) is assigned to the room(s) it separates: the aperture centroid must lie within 5 cm of a room face plane and inside that face bounding box. Any two rooms sharing an aperture form an access pair used later for circulation logic.

In [ ]:
all_apertures = windows + doors + entrance_doors
all_aperture_types = (
    ["window"]   * len(windows) +
    ["door"]     * len(doors) +
    ["entrance"] * len(entrance_doors)
)
apt_colors = {"window": "#4CC9F0", "door": "#F4A261", "entrance": "#E63946"}

# Centroid of each aperture face
apt_cents = []
for apt in all_apertures:
    c = Topology.Centroid(apt)
    apt_cents.append((Vertex.X(c), Vertex.Y(c), Vertex.Z(c)))

# Per-cell face data with bounding boxes
cell_face_planes_bbox = []
for c in cells:
    fd = []
    for f in (Topology.Faces(c) or []):
        fc  = Topology.Centroid(f)
        fp  = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn  = Face.Normal(f)
        fvs = Topology.Vertices(f) or []
        if fvs:
            xs = [Vertex.X(v) for v in fvs]
            ys = [Vertex.Y(v) for v in fvs]
            zs = [Vertex.Z(v) for v in fvs]
            bbox = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bbox = None
        fd.append((fp, fn, bbox))
    cell_face_planes_bbox.append(fd)

def in_bbox(pt, bbox, margin=0.05):
    if bbox is None:
        return True
    xmn, xmx, ymn, ymx, zmn, zmx = bbox
    return (xmn-margin <= pt[0] <= xmx+margin and
            ymn-margin <= pt[1] <= ymx+margin and
            zmn-margin <= pt[2] <= zmx+margin)

# Tight tolerances: aperture centroid must be very close to the face plane
# and inside (or just at) the face bounding box.
# This ensures each aperture matches only the two rooms it actually separates.
PLANE_TOL = 0.05  # 5 cm — was 15 cm; tighter prevents bleed into neighbouring rooms
apt_to_rooms = [set() for _ in all_apertures]
for ai, ac in enumerate(apt_cents):
    for ri, fd in enumerate(cell_face_planes_bbox):
        for fp, fn, bbox in fd:
            if perp_dist(ac, fp, fn) < PLANE_TOL and in_bbox(ac, bbox):
                apt_to_rooms[ai].add(ri)
                break

# Access pairs: rooms sharing an aperture
access_pairs = set()
for ai, rooms in enumerate(apt_to_rooms):
    if len(rooms) >= 2:
        room_list = sorted(rooms)
        for a in range(len(room_list)):
            for b in range(a + 1, len(room_list)):
                access_pairs.add((room_list[a], room_list[b]))

matched = sum(1 for r in apt_to_rooms if r)
print(f"All apertures: {len(all_apertures)}  "
      f"(windows={len(windows)}, doors={len(doors)}, entrance={len(entrance_doors)})")
print(f"Apertures matched to ≥1 room : {matched}")
print(f"Access pairs via apertures   : {len(access_pairs)}")
print("\nAccess pairs (room types):")
for i, j in sorted(access_pairs):
    ti = get_val(selector[i], "type")
    tj = get_val(selector[j], "type")
    print(f"  {ti} ↔ {tj}")

aperture_cluster = Cluster.ByTopologies(all_apertures)

Topology.Show(
    cell_cluster, aperture_cluster,
    faceColorKey="color",
    faceOpacity=0.15,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 15. Circulation Graph

One node per room; stairs are collapsed into two nodes — the **large/main stair** shaft and
the small **entrance stair** (see §7). Edges represent navigable connections:

1. **Doors / entrance doors** — rooms linked by real aperture access pairs (windows excluded).
   Stair doors attach to whichever stair node they actually touch.
2. **Open space** — non-enclosed rooms with facing wall surfaces and no shared wall.
3. **Explicit layout links** (geometry-grounded for this Old Layout):
   - Entrance stair ↔ **front corridor** (nearest corridor) and ↔ **small (front) balcony**
   - Large stair ↔ **top living room**, and that living room ↔ each **top-floor bedroom**
     (the top living room is the floor's hub)
4. **Constraint** — the small balcony is never connected to the large stair.

In [ ]:
# Circulation graph (Old Layout): movement via doors, open spaces, and explicit
# stair / balcony / top-floor links tailored to this layout.
circ_pair_set = set()

def same_level(i, j, z_tol=0.60):
    return abs(Vertex.Z(selector[i]) - Vertex.Z(selector[j])) <= z_tol

# 1. Door-based connections (real doors; windows excluded). Stair doors map to whichever
#    stair node (main/entrance) the door actually touches, via to_merged.
for ai, rooms in enumerate(apt_to_rooms):
    atype = all_aperture_types[ai]
    if atype not in {"door", "entrance"}:
        continue
    room_list = sorted(rooms)
    if len(room_list) < 2:
        continue
    for a in range(len(room_list)):
        for b in range(a + 1, len(room_list)):
            i, j = room_list[a], room_list[b]
            ti = get_val(selector[i], "type") or ""
            tj = get_val(selector[j], "type") or ""
            if ti != "Stair" and tj != "Stair" and not same_level(i, j):
                continue
            gi, gj = to_merged(i), to_merged(j)
            if gi != gj:
                circ_pair_set.add((min(gi, gj), max(gi, gj)))

# 2. Open-space adjacency - enclosed rooms never connect via open space
ENCLOSED = {"Bathroom", "Bedroom", "Storeroom"}
for i, j in open_pairs:
    ti = get_val(selector[i], "type") or ""
    tj = get_val(selector[j], "type") or ""
    if ti in ENCLOSED or tj in ENCLOSED:
        continue
    if not same_level(i, j):
        continue
    gi, gj = to_merged(i), to_merged(j)
    if gi != gj:
        circ_pair_set.add((min(gi, gj), max(gi, gj)))

# 3. Explicit, geometry-grounded stair / balcony / top-floor links
#    (entrance stair = stair_ext_idx, large/main stair = stair_main_idx from §7).
def _csq(a, b):
    return ((Vertex.X(a)-Vertex.X(b))**2 + (Vertex.Y(a)-Vertex.Y(b))**2 + (Vertex.Z(a)-Vertex.Z(b))**2)

balcony_idxs    = [i for i, s in enumerate(selector) if get_val(s, "type") == "Balcony"]
corridor_idxs   = [i for i, s in enumerate(selector) if get_val(s, "type") == "Corridor"]
livingroom_idxs = [i for i, s in enumerate(selector) if get_val(s, "type") == "Living room"]
bedroom_idxs    = [i for i, s in enumerate(selector) if get_val(s, "type") == "Bedroom"]

# small (front) balcony = smallest-area balcony; front corridor = corridor nearest the
# entrance-stair representative.
small_balcony  = min(balcony_idxs, key=lambda i: areas[i]) if balcony_idxs else None
front_corridor = (min(corridor_idxs, key=lambda i: _csq(selector[i], stair_ext_rep))
                  if corridor_idxs else None)

# top living room = the highest living room; top bedrooms = bedrooms on that same floor.
top_living = max(livingroom_idxs, key=lambda i: Vertex.Z(selector[i])) if livingroom_idxs else None
if top_living is not None:
    top_z        = Vertex.Z(selector[top_living])
    top_bedrooms = [i for i in bedroom_idxs if abs(Vertex.Z(selector[i]) - top_z) <= 0.60]
else:
    top_bedrooms = []

def add_edge(a_idx, b_idx):
    ga, gb = to_merged(a_idx), to_merged(b_idx)
    if ga != gb:
        circ_pair_set.add((min(ga, gb), max(ga, gb)))

# Entrance stair ↔ front corridor and ↔ small balcony
if front_corridor is not None:
    g = to_merged(front_corridor)
    circ_pair_set.add((min(g, stair_ext_idx), max(g, stair_ext_idx)))
if small_balcony is not None:
    g = to_merged(small_balcony)
    circ_pair_set.add((min(g, stair_ext_idx), max(g, stair_ext_idx)))

# Large stair ↔ top living room; top living room ↔ each top-floor bedroom
if top_living is not None:
    g = to_merged(top_living)
    circ_pair_set.add((min(g, stair_main_idx), max(g, stair_main_idx)))
    for i in top_bedrooms:
        add_edge(top_living, i)

# 4. Enforce constraints: the small balcony must NOT connect to the large stair.
if small_balcony is not None:
    g = to_merged(small_balcony)
    circ_pair_set.discard((min(g, stair_main_idx), max(g, stair_main_idx)))

circ_edges = [Edge.ByVertices([merged_nodes[i], merged_nodes[j]]) for i, j in sorted(circ_pair_set)]
g_access = Graph.ByVerticesEdges(merged_nodes, circ_edges)

for v in Graph.Vertices(g_access) or []:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    mi = min(range(len(merged_nodes)), key=lambda k: dist3(
        (vx, vy, vz),
        (Vertex.X(merged_nodes[k]), Vertex.Y(merged_nodes[k]), Vertex.Z(merged_nodes[k]))
    ))
    d_old = Topology.Dictionary(merged_nodes[mi])
    keys = list(Dictionary.Keys(d_old)) if d_old else []
    vals = list(Dictionary.Values(d_old)) if d_old else []
    if "size" in keys:
        vals[keys.index("size")] = merged_size(mi)
    else:
        keys.append("size")
        vals.append(merged_size(mi))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(keys, vals))

for e in Graph.Edges(g_access) or []:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [GRAPH_EDGE_WIDTH, GRAPH_EDGE_COLOR]))

print(f"Circulation graph - {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")
print(f"  entrance stair (node {stair_ext_idx}) ↔ front corridor "
      f"{'cell '+str(front_corridor) if front_corridor is not None else 'n/a'}, "
      f"small balcony {'cell '+str(small_balcony) if small_balcony is not None else 'n/a'}")
print(f"  large stair    (node {stair_main_idx}) ↔ top living room "
      f"{'cell '+str(top_living) if top_living is not None else 'n/a'}; "
      f"top bedrooms {sorted(top_bedrooms)}")

Topology.Show(
    [cell_cluster, g_access],
    faceColorKey="color", faceOpacity=0.12,
    edgeColor=GRAPH_EDGE_COLOR, edgeWidth=0.5,
    edgeWidthKey="width", edgeColorKey="color",
    vertexSizeKey=GRAPH_NODE_SIZE_KEY, vertexColorKey=GRAPH_NODE_COLOR_KEY,
    vertexLabelKey=GRAPH_NODE_LABEL_KEY, showVertexLabel=True, showVertices=True,
    backgroundColor=GRAPH_BG_LIGHT, width=1000, height=1000,
    renderer=renderer
)

## 16. Room–Aperture Bipartite Graph

Two sets of nodes: **rooms** (coloured by type, sized by surface area; each stair = single
node) and **apertures** (windows blue, doors orange, entrance red). Each gray edge connects
an aperture to the room it belongs to.

Overlaid in **stair purple** are the top-floor circulation links from §15: the large/main
stair ↔ the top living room, and that living room ↔ each top-floor bedroom.

In [ ]:
from collections import defaultdict

apt_colors = {"window": "#4CC9F0", "door": "#F4A261", "entrance": "#E63946"}

# Room vertices: one per merged node (stair = 1 node), area-scaled size
room_verts = []
for mi, mn in enumerate(merged_nodes):
    v = Vertex.ByCoordinates(Vertex.X(mn), Vertex.Y(mn), Vertex.Z(mn))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(
        ["size", "color"],
        [merged_size(mi), get_val(mn, "color") or "gray"]
    ))
    room_verts.append(v)

# Aperture vertices — one per unique aperture face
aperture_verts = []
for apt, atype in zip(all_apertures, all_aperture_types):
    ac = Topology.Centroid(apt)
    v  = Vertex.ByCoordinates(Vertex.X(ac), Vertex.Y(ac), Vertex.Z(ac))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(
        ["size", "color"], [7, apt_colors[atype]]
    ))
    aperture_verts.append(v)

# Bipartite edges: merged room ↔ aperture
edges_ra = []
merged_to_windows   = defaultdict(int)
merged_to_doors     = defaultdict(int)
merged_to_entrances = defaultdict(int)

for ai, rooms in enumerate(apt_to_rooms):
    atype  = all_aperture_types[ai]
    av     = aperture_verts[ai]
    seen_merged = set()
    for ri in sorted(rooms):
        mi = to_merged(ri)
        if mi in seen_merged:
            continue
        seen_merged.add(mi)
        e = Edge.ByVertices([room_verts[mi], av])
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [GRAPH_EDGE_WIDTH, GRAPH_EDGE_COLOR]))
        edges_ra.append(e)
        if atype == "window":
            merged_to_windows[mi] += 1
        elif atype == "door":
            merged_to_doors[mi] += 1
        else:
            merged_to_entrances[mi] += 1

# Top-floor circulation links (mirrors §15), drawn as room↔room edges in stair purple:
# the large/main stair ↔ the top living room, and that living room ↔ each top-floor bedroom.
livingroom_idxs = [i for i, s in enumerate(selector) if get_val(s, "type") == "Living room"]
bedroom_idxs    = [i for i, s in enumerate(selector) if get_val(s, "type") == "Bedroom"]
room_room_edges = []
if livingroom_idxs:
    _top_living   = max(livingroom_idxs, key=lambda i: Vertex.Z(selector[i]))
    _top_z        = Vertex.Z(selector[_top_living])
    _top_bedrooms = [i for i in bedroom_idxs if abs(Vertex.Z(selector[i]) - _top_z) <= 0.60]
    _lr           = to_merged(_top_living)

    _rr_pairs = set()
    _rr_pairs.add((min(_lr, stair_main_idx), max(_lr, stair_main_idx)))   # large stair ↔ top living room
    for i in _top_bedrooms:
        gi = to_merged(i)
        if gi != _lr:
            _rr_pairs.add((min(_lr, gi), max(_lr, gi)))                   # top living room ↔ each top bedroom

    for a, b in sorted(_rr_pairs):
        e = Edge.ByVertices([room_verts[a], room_verts[b]])
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [GRAPH_EDGE_WIDTH, "#8338EC"]))
        room_room_edges.append(e)
    edges_ra.extend(room_room_edges)
    print(f"Top-floor room links added: large stair↔livingroom[{_top_living}] + "
          f"{len(_rr_pairs)-1} livingroom↔bedroom  (top bedrooms {sorted(_top_bedrooms)})")

print("Windows per room:")
for mi in sorted(merged_to_windows):
    print(f"  {get_val(merged_nodes[mi],'label')}: {merged_to_windows[mi]}")
print("Doors per room:")
for mi in sorted(merged_to_doors):
    print(f"  {get_val(merged_nodes[mi],'label')}: {merged_to_doors[mi]}")
print("Entrance doors per room:")
for mi in sorted(merged_to_entrances):
    print(f"  {get_val(merged_nodes[mi],'label')}: {merged_to_entrances[mi]}")
print(f"Bipartite edges: {len(edges_ra)}  (incl. {len(room_room_edges)} top-floor room links)")

In [ ]:
bipartite = room_verts + aperture_verts + edges_ra

Topology.Show(
    [cell_cluster, aperture_cluster] + bipartite,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor=GRAPH_EDGE_COLOR, edgeWidth=0.5,
    vertexColor="black",
    vertexSizeKey=GRAPH_NODE_SIZE_KEY, vertexColorKey=GRAPH_NODE_COLOR_KEY,
    edgeWidthKey="width", edgeColorKey="color",
    showVertices=True,
    backgroundColor=GRAPH_BG_DARK,
    width=900, height=700,
    renderer=renderer
)

## 17. Final Summary

This model captures a typical Brooklyn brownstone organization as a stacked, circulation-driven house: vertically connected by a central stair and horizontally organized by room-to-room wall adjacency on each level.

The graph outputs highlight three complementary relationships:
- Horizontal adjacency (shared walls on the same level)
- Vertical adjacency (rooms directly above or below across floor slabs)
- Circulation connectivity (door-based movement plus stair-corridor structure)

Together, these graphs describe how privacy, service spaces, and movement are structured in a narrow multi-level townhouse layout.